# Three-SMU live scan

This notebook uses the same audited generator as the CLI. It does not import QCoDeS directly. Real connection and writes require manually changing both authorization flags after reviewing the one local `hardware.local.toml`.

In [ ]:
from pathlib import Path
from queue import SimpleQueue
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

from attodry_control.three_smu import ThreeSmuSession
from attodry_control.three_smu_config import (
    load_three_smu_operation_config, validate_plan_targets,
)


In [ ]:
CONFIG_TOML = Path('../config/hardware.local.toml')
AUTHORIZE_WRITES = False
AUTHORIZE_STATUS_CONSUMPTION = False

operation = load_three_smu_operation_config(CONFIG_TOML)
hardware = operation.hardware
plan = operation.plan
points = validate_plan_targets(hardware, plan)
print(f'Validated {len(points)} points without opening hardware.')


In [ ]:
elapsed, bias_current = [], []
sample_queue = SimpleQueue()

def display_queued_samples():
    """Plot only formal samples already published to the in-memory queue."""
    while not sample_queue.empty():
        sample = sample_queue.get()
        elapsed.append(sample.elapsed_s)
        bias_current.append(sample.readings['smu_bias'].reading.current_a)
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6.4, 4.0), constrained_layout=True)
        ax.plot(elapsed, bias_current, marker='o')
        ax.set(xlabel='Elapsed time (s)', ylabel='Bias current (A)', title='Live Three-SMU scan')
        ax.grid(True, alpha=0.25)
        display(fig)
        plt.close(fig)

with ThreeSmuSession.open(
    hardware, plan, authorize_writes=AUTHORIZE_WRITES,
    authorize_status_consumption=AUTHORIZE_STATUS_CONSUMPTION,
) as session:
    try:
        for _ in session.run(
            output_dir=operation.output_directory, run_name=operation.run_name,
            note=operation.note, config_path=operation.config_path,
            on_sample=sample_queue.put,
        ):
            display_queued_samples()
    finally:
        # A retained problem sample reaches this queue before the session aborts.
        display_queued_samples()
print(f'Run directory: {session.last_run_dir}')
